## Part 1 -- Currency Exchange Problem

In [7]:
import numpy as np
import cvxpy as cp

# --- Data Setup for the Challenging Problem ---
currencies = ['USD', 'EUR', 'GBP', 'JPY', 'CHF'] 
n = len(currencies)

# 1. Initial Holdings (in native units) - SCARCE & DIVERSE
c_init = np.array([100000.0,   # USD
                   80000.0,    # EUR
                   50000.0,    # GBP
                   1000000.0,  # JPY (REDUCED! Cannot cover requirement alone)
                   100000.0])       # CHF

# 2. Required Holdings (in native units) - ONLY CHF
c_req = np.array([50000.0,        # USD
                  60000.0,        # EUR
                  60000.0,        # GBP
                  600000.0,        # JPY
                  150000.0])  # CHF

# 3. MANIPULATED Exchange Rate Matrix (R[i, j])
R = np.array([[1.00,   0.90,   0.80,   0.009,  0.99],   # USD
              [1.10,   1.00,   0.90,   0.010,  1.08],   # EUR
              [1.25,   1.11,   1.00,   0.011,  1.22],   # GBP
              [111.0,  100.0,  90.0,   1.00,   120.0],  # JPY
              [1.00,   0.91,   0.80,   0.008,  1.00]])  # CHF

# 4. MANIPULATED Transaction Cost Matrix (Delta[i, j])
# High cost (2%) for any trade involving CHF. Low cost (0.1%) for others.
Delta = np.array([[0.0000, 0.0010, 0.0010, 0.0010, 0.0200],
                  [0.0010, 0.0000, 0.0010, 0.0010, 0.0200],
                  [0.0010, 0.0010, 0.0000, 0.0010, 0.0200],
                  [0.0010, 0.0010, 0.0010, 0.0000, 0.0200],
                  [0.0200, 0.0200, 0.0200, 0.0200, 0.0000]])



In [8]:
# ==============================================================
# OPTIMAL CONSUMPTION & PORTFOLIO PROBLEM
# Lecture 16, IEDA 4000H, HKUST, Nov 4, 2025
# Instructor: Zijie Zhou
# ==============================================================

import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt

# -----------------------------
# 1. Problem Parameters
# -----------------------------
T = 40                     # years until retirement
n = 3                      # assets: stocks, bonds, cash
rho = 0.5                  # risk aversion parameter (ρ < 1)
beta = 0.9                 # bequest importance

# Asset expected returns (annual)
mu = np.array([0.08, 0.04, 0.02])  # [stocks, bonds, cash]

# Covariance matrix
Sigma = np.array([
    [0.04,   0.01,   0.0001],
    [0.01,   0.0225, 0.0001],
    [0.0001, 0.0001, 0.0001]
])

# Labor income over time
y = np.zeros(T)
y[:10] = 60000 + 2000 * np.arange(10)   # growth phase
y[10:30] = 80000                        # peak
y[30:] = 70000                          # pre-retirement

k0 = 10000                 # initial wealth
mu_rf = mu[2]              # risk-free rate ≈ cash return

# -----------------------------
# 2. Scale for numerical stability
# -----------------------------
scale_factor = 10000
k0_scaled = k0 / scale_factor
y_scaled = y / scale_factor

# -----------------------------
# 3. Monte Carlo Return Scenarios
# -----------------------------
N_scen = 5000
np.random.seed(42)

# Generate log-normal gross returns: R_t = exp(r_t) where r_t ~ N(mu, Sigma)
# We simulate gross returns directly
R_scen = np.zeros((T, n, N_scen))
for t in range(T):
    log_returns = np.random.multivariate_normal(mu, Sigma, N_scen)
    R_scen[t, :, :] = np.exp(log_returns).T  # shape: (n, N_scen)

# -----------------------------
# 4. CVXPY Variables
# -----------------------------
c = cp.Variable((T, N_scen))        # consumption
h = cp.Variable((T, n, N_scen))     # investment in each asset
k = cp.Variable((T+1, N_scen))      # wealth at start of each period

# -----------------------------
# 5. Constraints
# -----------------------------
constraints = []

# Initial wealth
constraints += [k[0, :] == k0_scaled]

# Wealth dynamics: k_{t+1} = k_t - c_t + y_t + h_t @ R_t
for t in range(T):
    portfolio_return = cp.sum(cp.multiply(h[t, :, :], R_scen[t, :, :]), axis=0)
    constraints += [
        k[t+1, :] == k[t, :] - c[t, :] + y_scaled[t] + portfolio_return
    ]

# Non-negativity
constraints += [c >= 0, k >= 0]
# Optional: no short-selling
constraints += [h >= 0]

# -----------------------------
# 6. Objective: Expected Utility
# -----------------------------
# E[ ∑_{t=0}^{T-1} c_t^ρ/ρ + β k_T^ρ/ρ ]
util_consumption = cp.sum(cp.power(c, rho)) / rho
util_bequest = beta * cp.sum(cp.power(k[T, :], rho)) / rho
expected_utility = (util_consumption + util_bequest) / N_scen

objective = cp.Maximize(expected_utility)

# -----------------------------
# 7. Solve
# -----------------------------
prob = cp.Problem(objective, constraints)
print("Solving... (this may take 10–30 seconds)")
prob.solve(
    solver=cp.SCS,
    max_iters=2000,
    eps=1e-8,
    verbose=True
)

print(f"\nStatus: {prob.status}")
print(f"Optimal expected utility (scaled): {prob.value:.6f}")

# -----------------------------
# 8. Extract & Unscale Results
# -----------------------------
c_opt = c.value * scale_factor
k_opt = k.value * scale_factor
h_opt = h.value * scale_factor

# Average paths
c_mean = np.mean(c_opt, axis=1)
k_mean = np.mean(k_opt, axis=1)
h_mean = np.mean(h_opt, axis=1)  # average dollar allocation per asset

# Portfolio weights (average over time and scenarios)
w_mean = np.zeros((T, n))
for t in range(T):
    total_wealth = k_mean[t]
    if total_wealth > 1:
        w_mean[t, :] = h_mean[t, :] / total_wealth
    else:
        w_mean[t, :] = np.nan

# -----------------------------
# 9. Plot Results
# -----------------------------
years = np.arange(T)

plt.figure(figsize=(15, 10))

# Plot 1: Consumption vs Income
plt.subplot(2, 2, 1)
plt.plot(years, c_mean, label='Optimal Consumption', marker='o', markersize=3)
plt.plot(years, y, label='Labor Income', linestyle='--', color='gray')
plt.title('Consumption and Income Over Time')
plt.xlabel('Year')
plt.ylabel('USD')
plt.legend()
plt.grid(True)

# Plot 2: Wealth Trajectory
plt.subplot(2, 2, 2)
plt.plot(np.arange(T+1), k_mean, label='Wealth', color='green')
plt.title('Wealth Trajectory')
plt.xlabel('Year')
plt.ylabel('USD')
plt.yscale('log')
plt.grid(True)

# Plot 3: Portfolio Weights
plt.subplot(2, 2, 3)
asset_names = ['Stocks', 'Bonds', 'Cash']
for i in range(n):
    plt.plot(years, w_mean[:, i], label=asset_names[i])
plt.title('Average Portfolio Weights')
plt.xlabel('Year')
plt.ylabel('Weight')
plt.legend()
plt.ylim(0, 1)
plt.grid(True)

# Plot 4: Consumption Smoothing
plt.subplot(2, 2, 4)
plt.plot(years, c_mean, label='Consumption')
plt.axhline(np.mean(c_mean), color='red', linestyle=':', label=f'Avg = ${np.mean(c_mean):,.0f}')
plt.title('Smoothed Consumption')
plt.xlabel('Year')
plt.ylabel('USD')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# -----------------------------
# 10. Summary Output
# -----------------------------
print("\n" + "="*60)
print("SUMMARY OF OPTIMAL POLICY")
print("="*60)
print(f"Initial Wealth:        ${k0:,.0f}")
print(f"Average Annual Income: ${np.mean(y):,.0f}")
print(f"Average Consumption:   ${np.mean(c_mean):,.0f}")
print(f"Final Average Wealth:  ${k_mean[-1]:,.0f}")
print(f"Bequest Utility Weight (β): {beta}")
print(f"Risk Aversion (ρ):     {rho}")
print(f"Scenarios Used:        {N_scen}")
print("="*60)


Solving... (this may take 10–30 seconds)
                                     CVXPY                                     
                                     v1.6.0                                    
(CVXPY) Nov 05 11:37:37 AM: Your problem has 1005000 variables, 1210000 constraints, and 0 parameters.
(CVXPY) Nov 05 11:37:37 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Nov 05 11:37:37 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Nov 05 11:37:37 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Nov 05 11:37:37 AM: Your problem is compiled with the CPP canonicalization backend.
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
(CVXPY) Nov 05 11:37

/home/ngdavian/anaconda3/lib/python3.12/site-packages/cvxpy/problems/problem.py:1481: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


ValueError: could not broadcast input array from shape (5000,) into shape (3,)

## Part 2 -- Robust Currency Exchange Problem Under Box Uncertainty

In [9]:
import numpy as np
import cvxpy as cp

# --- Data Setup ---
currencies = ['USD', 'EUR', 'GBP', 'JPY', 'CHF'] 
n = len(currencies)

# 1. Initial Holdings
c_init = np.array([100000.0,   # USD
                   80000.0,    # EUR
                   50000.0,    # GBP
                   1000000.0,  # JPY
                   100000.0])  # CHF

# 2. Required Holdings
c_req = np.array([50000.0,     # USD
                  60000.0,     # EUR
                  60000.0,     # GBP
                  600000.0,    # JPY
                  150000.0])   # CHF


# 3. NOMINAL Exchange Rate Matrix
R_nom = np.array([[1.00,   0.90,   0.80,   0.009,  0.99],
                  [1.10,   1.00,   0.90,   0.010,  1.08],
                  [1.25,   1.11,   1.00,   0.011,  1.22],
                  [111.0,  100.0,  90.0,   1.00,   120.0],
                  [1.00,   0.91,   0.80,   0.008,  1.00]])

# 4. NOMINAL Transaction Cost Matrix
Delta_nom = np.array([[0.0000, 0.0010, 0.0010, 0.0010, 0.0200],
                      [0.0010, 0.0000, 0.0010, 0.0010, 0.0200],
                      [0.0010, 0.0010, 0.0000, 0.0010, 0.0200],
                      [0.0010, 0.0010, 0.0010, 0.0000, 0.0200],
                      [0.0200, 0.0200, 0.0200, 0.0200, 0.0000]])

# --- Define Uncertainty Radii ---
# Assume 2% uncertainty for exchange rates
rho_R = 0.02 * R_nom
# Avoid negative radii for zero elements (like diagonals)
rho_R = np.maximum(rho_R, 0.0)

# Assume 10% uncertainty for transaction costs
rho_Delta = 0.10 * Delta_nom
rho_Delta = np.maximum(rho_Delta, 0.0)

print("Exchange Rate Uncertainty Radii (rho_R):")
print(rho_R)
print("\nTransaction Cost Uncertainty Radii (rho_Delta):")
print(rho_Delta)

Exchange Rate Uncertainty Radii (rho_R):
[[2.00e-02 1.80e-02 1.60e-02 1.80e-04 1.98e-02]
 [2.20e-02 2.00e-02 1.80e-02 2.00e-04 2.16e-02]
 [2.50e-02 2.22e-02 2.00e-02 2.20e-04 2.44e-02]
 [2.22e+00 2.00e+00 1.80e+00 2.00e-02 2.40e+00]
 [2.00e-02 1.82e-02 1.60e-02 1.60e-04 2.00e-02]]

Transaction Cost Uncertainty Radii (rho_Delta):
[[0.     0.0001 0.0001 0.0001 0.002 ]
 [0.0001 0.     0.0001 0.0001 0.002 ]
 [0.0001 0.0001 0.     0.0001 0.002 ]
 [0.0001 0.0001 0.0001 0.     0.002 ]
 [0.002  0.002  0.002  0.002  0.    ]]


In [10]:
# ==============================================================
# LINEAR PROGRAMMING APPROXIMATION OF DYNAMIC PROGRAMMING
# Optimal Consumption Problem - Lecture 16, IEDA 4000H
# Uses Piecewise-Linear Utility + Scenario Tree
# ==============================================================

import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt

# -----------------------------
# 1. Parameters (from lecture)
# -----------------------------
T = 40
n = 3
rho = 0.5
beta = 0.9

mu = np.array([0.08, 0.04, 0.02])
Sigma = np.array([
    [0.04,   0.01,   0.0001],
    [0.01,   0.0225, 0.0001],
    [0.0001, 0.0001, 0.0001]
])

y = np.zeros(T)
y[:10] = 60000 + 2000 * np.arange(10)
y[10:30] = 80000
y[30:] = 70000

k0 = 10000
scale = 10000
k0_s = k0 / scale
y_s = y / scale

# -----------------------------
# 2. Monte Carlo Scenarios
# -----------------------------
N_scen = 2000
np.random.seed(42)
R = np.exp(np.random.multivariate_normal(mu, Sigma, size=(T, N_scen)))  # gross returns
R = R.T  # shape: (n, T, N_scen) → now (T, n, N_scen)

# -----------------------------
# 3. Piecewise-Linear Utility Grid
# -----------------------------
# Grid for consumption and bequest (in scaled units)
c_max = 15.0  # max consumption ~$150k
k_max = 100.0 # max wealth ~$1M
num_segments = 50

c_grid = np.linspace(0, c_max, num_segments + 1)
k_grid = np.linspace(0, k_max, num_segments + 1)

# Precompute u(c) = c^rho / rho on grid
u_c = c_grid**rho / rho
u_k = k_grid**rho / rho

# -----------------------------
# 4. CVXPY Variables
# -----------------------------
c = cp.Variable((T, N_scen))           # actual consumption
k = cp.Variable((T+1, N_scen))         # wealth
h = cp.Variable((T, n, N_scen))        # investment

# PWL variables: lambda weights for convex combination
lam_c = cp.Variable((T, N_scen, num_segments + 1))  # for consumption
lam_k = cp.Variable((1, N_scen, num_segments + 1))  # for bequest

# -----------------------------
# 5. Constraints
# -----------------------------
constraints = []

# Initial wealth
constraints += [k[0, :] == k0_s]

# Wealth dynamics
for t in range(T):
    port_return = cp.sum(cp.multiply(h[t, :, :], R[t, :, :]), axis=0)
    constraints += [
        k[t+1, :] == k[t, :] - c[t, :] + y_s[t] + port_return
    ]

# Non-negativity
constraints += [c >= 0, k >= 0, h >= 0]

# --- PWL for consumption utility ---
for t in range(T):
    for s in range(N_scen):
        # c[t,s] = sum λ_j * c_grid[j]
        constraints += [c[t, s] == cp.sum(cp.multiply(lam_c[t, s, :], c_grid))]
        # sum λ = 1, λ >= 0, at most 2 adjacent λ > 0 (SOS2)
        constraints += [cp.sum(lam_c[t, s, :]) == 1]
        constraints += [lam_c[t, s, :] >= 0]
        # SOS2: use binary variables to enforce adjacent support
        delta = cp.Variable(num_segments, boolean=True)
        constraints += [lam_c[t, s, :-1] <= delta]
        constraints += [lam_c[t, s, 1:] <= delta]
        constraints += [cp.sum(delta) <= 2]

# --- PWL for bequest utility ---
for s in range(N_scen):
    constraints += [k[T, s] == cp.sum(cp.multiply(lam_k[0, s, :], k_grid))]
    constraints += [cp.sum(lam_k[0, s, :]) == 1]
    constraints += [lam_k[0, s, :] >= 0]
    delta_k = cp.Variable(num_segments, boolean=True)
    constraints += [lam_k[0, s, :-1] <= delta_k]
    constraints += [lam_k[0, s, 1:] <= delta_k]
    constraints += [cp.sum(delta_k) <= 2]

# -----------------------------
# 6. Objective: Expected PWL Utility
# -----------------------------
util_cons = cp.sum(cp.sum(cp.multiply(lam_c, u_c), axis=2))
util_beq  = beta * cp.sum(cp.sum(cp.multiply(lam_k, u_k), axis=2))
objective = cp.Maximize((util_cons + util_beq) / N_scen)

# -----------------------------
# 7. Solve (MILP → but very fast with Gurobi/CBC)
# -----------------------------
prob = cp.Problem(objective, constraints)
print("Solving LP approximation of DP (this may take 1-3 minutes)...")
prob.solve(solver=cp.GUROBI, verbose=True)  # or cp.CBC, cp.GLPK_MI

print(f"\nStatus: {prob.status}")
print(f"Objective (scaled): {prob.value:.6f}")

# -----------------------------
# 8. Extract Results
# -----------------------------
c_opt = c.value * scale
k_opt = k.value * scale
h_opt = h.value * scale

c_mean = np.mean(c_opt, axis=1)
k_mean = np.mean(k_opt, axis=1)

# -----------------------------
# 9. Plot
# -----------------------------
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(c_mean, label='Avg Consumption (LP)', marker='o')
plt.plot(y, label='Labor Income', linestyle='--', alpha=0.7)
plt.title('Consumption Path (LP Approximation)')
plt.xlabel('Year')
plt.ylabel('USD')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(k_mean, label='Avg Wealth', color='green')
plt.title('Wealth Trajectory')
plt.xlabel('Year')
plt.ylabel('USD')
plt.yscale('log')
plt.grid(True)

plt.tight_layout()
plt.show()

ValueError: shape mismatch: objects cannot be broadcast to a single shape.  Mismatch is between arg 0 with shape (3, 2000) and arg 1 with shape (2000, 40).